# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`.

### Dataset Source
The dataset is provided as a Croissant schema:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}\nIdentifier: {meta.identifier}")

## 2. Data Overview
Review available record sets and their fields with corresponding `@id` references.

In [ ]:
# Get all record sets in the dataset by @id
record_sets = dataset.metadata.record_set
print("Record sets in dataset:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '[No name]')}")

# For each record set, list its fields and corresponding @id
for rs in record_sets:
    print(f"\nFields in Record Set: {rs['@id']} ({rs.get('name', '[No name]')})")
    if 'field' in rs:
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"  - {field['@id']}: {field.get('name', '[No name]')}")
            else:
                # Some schemas may only have string @id references
                print(f"  - {field}")
    else:
        print("  [No fields declared]")

## 3. Data Extraction
Load data from a specific record set using its `@id`. The main data table contains one record set; we extract all rows and show the available columns using their Croissant `@id`.

In [ ]:
# Identify record set IDs (use @id references from previous overview)
main_record_set_ids = [rs['@id'] for rs in dataset.metadata.record_set]

# For this dataset there is likely only one main record set, otherwise adjust accordingly
dataframes = {}
for record_set_id in main_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecord set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filter, normalize, and aggregate using fields referenced by `@id`. We select a numeric field for filtering, normalize it, and group records by a categorical field.

Fields used below should be referenced by their `@id` as listed above.

In [ ]:
# Example field @ids: replace with the actual ones from your schema, e.g.:
# record_set_id = "http://mlcommons.org/croissant/frontiers/colorectal-cancer-main-table"
# numeric_field_id = "http://mlcommons.org/croissant/frontiers/age-at-crc-diagnosis"
# group_field_id = "http://mlcommons.org/croissant/frontiers/msi-h-status"

# Use the one main record set in this dataset
record_set_id = main_record_set_ids[0]
df = dataframes[record_set_id]

# Show all available columns to choose appropriate fields
print("Available columns (Croissant @id):")
print(df.columns.tolist())

# For demonstration, suppose age at CRC diagnosis is a numeric field @id:
numeric_field = None
group_field = None
# Heuristics: look for 'age' or 'Age' in columns, and 'msi' or 'MSI' for grouping
for col in df.columns:
    if 'age' in col.lower() and not numeric_field:
        numeric_field = col
    if ('msi' in col.lower() or 'status' in col.lower() or 'sex' in col.lower()) and not group_field:
        group_field = col

if numeric_field is None:
    print("No numeric field found for demonstration.")
else:
    threshold = 50  # Example threshold; adjust as per your data analysis goals
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships of key numeric and categorical fields. Below is a histogram of a numeric field and a boxplot grouped by a categorical field (all referenced by Croissant `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
We have demonstrated how to load the FAIR² dataset using the `mlcroissant` API, referenced entities by their Croissant `@id`, and explored, filtered, and visualized clinical data for further analysis.

**Key Findings:**
- The dataset contains detailed clinicopathological variables for 77 cancer survivors with second primary colorectal cancer.
- Key fields (by `@id`) such as age and MSI-H status can be used for filtering and group-wise analysis.
- Filtering and normalization illustrated typical pre-processing steps, and visualizations enabled quick summary of variable distributions.

For more advanced analysis, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/) and further leverage the provided field and record set `@id`s for reproducible workflows.